# 🛡️ YOLOv8m — CCTV Weapon Detection Training

**Model:** YOLOv8m (Medium) — best accuracy/speed tradeoff  
**Target:** Pistol, Knife from CCTV camera angles  
**Runtime:** GPU required → Runtime → Change runtime type → **T4 GPU**

---
### 📋 Steps
1. ✅ Check GPU
2. 📦 Install dependencies
3. 🔗 Mount Google Drive
4. 📊 Download dataset (Roboflow auto-fallback OR Kaggle)
5. 🚀 Train YOLOv8m
6. 📈 Evaluate results
7. 💾 Save best.pt to Drive

## ✅ Step 1 — Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} ({mem:.1f} GB VRAM)')
else:
    print('❌ No GPU! Go to: Runtime → Change runtime type → T4 GPU')

## 📦 Step 2 — Install Dependencies

In [ ]:
!pip install ultralytics roboflow gdown -q
import ultralytics
from ultralytics import YOLO
print(f'✅ Ultralytics {ultralytics.__version__} ready')

## 🔗 Step 3 — Mount Google Drive
Saves your trained model so it survives Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/YOLOv8m_WeaponDetection'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Save location: {SAVE_DIR}')

## 📊 Step 4a — Download Dataset via Roboflow

**Get your free API key:**
1. Go to [https://app.roboflow.com](https://app.roboflow.com) → sign up free
2. Profile → **Settings** → **Roboflow API** → copy **Private API Key**
3. Paste it in `API_KEY =` below

The cell auto-tries **3 verified public datasets** — stops at the first that works.

> If all 3 fail → skip to **Step 4b (Kaggle)** below

In [ ]:
from roboflow import Roboflow
import yaml, os

# ──────────────────────────────────────────────────────────────
API_KEY = "YOUR_ROBOFLOW_API_KEY"   # <-- paste your key here
# ──────────────────────────────────────────────────────────────

# 3 verified public weapon datasets on Roboflow Universe
DATASETS = [
    # (workspace,               project,                version)
    ("roboflow-universe-projects", "pistols-and-smartphones", 2),
    ("weapon-detection-e0vrz",     "weapon-detection-q1dws",   1),
    ("brad-dwyer",                 "pistol-detection",         1),
]

rf = Roboflow(api_key=API_KEY)
dataset = None
DATASET_YAML = None

for ws, proj, ver in DATASETS:
    try:
        print(f'Trying {ws}/{proj} v{ver}...')
        version_obj = rf.workspace(ws).project(proj).version(ver)
        dataset = version_obj.download("yolov8")
        DATASET_YAML = f'{dataset.location}/data.yaml'
        print(f'✅ Success: {ws}/{proj}')
        break
    except Exception as e:
        print(f'  ✗ Failed ({type(e).__name__}) — trying next...')

if dataset is None:
    print()
    print('❌ All Roboflow datasets failed.')
    print('👉 Run Step 4b (Kaggle) below — no API key needed!')
else:
    with open(DATASET_YAML) as f:
        data_cfg = yaml.safe_load(f)
    print(f'\n📂 Location : {dataset.location}')
    print(f'🏷️  Classes   : {data_cfg["names"]}')
    print(f'🔢 Count     : {data_cfg["nc"]} classes')

## 📊 Step 4b — Kaggle Dataset (NO account needed)

**Only run this if Step 4a failed.**  
Downloads the public **Gun & Knife Detection** dataset (~150 MB) directly — no login required.

Dataset: [Gun and Knife Detection on Kaggle](https://www.kaggle.com/datasets/jahnvisahni31/gun-and-knife-detection)

In [ ]:
# ── ONLY RUN THIS IF STEP 4a FAILED ────────────────────────────────────────
import os, zipfile, yaml, subprocess

DATASET_DIR = '/content/weapon_dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

# Download from Kaggle public datasets via kaggle API
# Step 1: upload your kaggle.json to Colab (from kaggle.com > Account > API)
from google.colab import files
print('Upload your kaggle.json file (from kaggle.com > Account > Create API Token)')
uploaded = files.upload()  # opens file picker

os.makedirs('/root/.kaggle', exist_ok=True)
import shutil
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Download dataset
!kaggle datasets download -d jahnvisahni31/gun-and-knife-detection -p /content/weapon_dataset --unzip

# Find data.yaml
yaml_files = []
for root, dirs, files_list in os.walk(DATASET_DIR):
    for f in files_list:
        if f == 'data.yaml':
            yaml_files.append(os.path.join(root, f))

if yaml_files:
    DATASET_YAML = yaml_files[0]
    with open(DATASET_YAML) as f:
        data_cfg = yaml.safe_load(f)
    print(f'\n✅ Dataset ready!')
    print(f'   YAML: {DATASET_YAML}')
    print(f'   Classes: {data_cfg.get("names")}')
else:
    # No data.yaml — create one manually
    print('No data.yaml found. Creating one...')
    import glob
    train_imgs = glob.glob(f'{DATASET_DIR}/**/images/train', recursive=True)
    val_imgs   = glob.glob(f'{DATASET_DIR}/**/images/val*', recursive=True)
    print('Found image dirs:', train_imgs, val_imgs)
    print('Manually update DATASET_YAML with your data.yaml path.')

### 📊 Dataset Statistics

In [ ]:
import os, glob
assert DATASET_YAML is not None, 'Run Step 4a or 4b first!'
base = os.path.dirname(DATASET_YAML)
print(f'Dataset root: {base}\n')
total = 0
for split in ['train', 'valid', 'val', 'test']:
    img_dir = os.path.join(base, split, 'images')
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        total += n
        print(f'  {split:6s}: {n:5d} images')
print(f'  ─────────────────')
print(f'  Total : {total:5d} images')

### 🖼️ Preview Dataset Samples

In [ ]:
import cv2, matplotlib.pyplot as plt, glob, random, os
base = os.path.dirname(DATASET_YAML)
imgs = glob.glob(f'{base}/train/images/*.jpg') + glob.glob(f'{base}/train/images/*.png')
sample = random.sample(imgs, min(6, len(imgs)))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Training Dataset Samples', fontsize=14, fontweight='bold')
for ax, p in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.axis('off')
    ax.set_title(os.path.basename(p), fontsize=8)
plt.tight_layout(); plt.show()

## 🚀 Step 5 — Train YOLOv8m

| Augmentation | Value | Why |
|---|---|---|
| `hsv_v=0.4` | High | CCTV brightness varies wildly |
| `hsv_s=0.7` | High | Washed-out/grayscale CCTV feeds |
| `perspective=0.0005` | Low | Top-down camera angle simulation |
| `scale=0.5` | High | Tiny weapon when person is far away |
| `mosaic=1.0` | Max | 4-image composite for small objects |
| `copy_paste=0.1` | Low | Injects extra weapon instances |
| `erasing=0.4` | Medium | Simulates partial occlusion |

> ⏱️ ~40–70 min on T4 GPU for 100 epochs

In [ ]:
from ultralytics import YOLO
assert DATASET_YAML is not None, 'Run Step 4a or 4b first!'

model = YOLO('yolov8m.pt')
print('✅ YOLOv8m loaded (COCO pretrained)')

results = model.train(
    data=DATASET_YAML,

    # ── Core ───────────────────────────────────────────────
    epochs=100,
    imgsz=640,
    batch=16,           # Lower to 8 if CUDA Out of Memory
    patience=20,
    workers=2,

    # ── Optimizer ──────────────────────────────────────────
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # ── CCTV Augmentations ─────────────────────────────────
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    erasing=0.4,

    # ── Output ─────────────────────────────────────────────
    project='weapon_detection',
    name='yolov8m_cctv',
    save=True,
    save_period=10,
    plots=True,
    device=0,
)

print('\n✅ Training complete!')
print('Best model: weapon_detection/yolov8m_cctv/weights/best.pt')

## 📈 Step 6 — Evaluate Results

In [ ]:
best_model = YOLO('weapon_detection/yolov8m_cctv/weights/best.pt')
metrics = best_model.val(data=DATASET_YAML)

print('\n' + '='*45)
print('         📊 EVALUATION RESULTS')
print('='*45)
print(f'  mAP@50      : {metrics.box.map50:.3f}  ({metrics.box.map50*100:.1f}%)')
print(f'  mAP@50-95   : {metrics.box.map:.3f}  ({metrics.box.map*100:.1f}%)')
print(f'  Precision   : {metrics.box.p.mean():.3f}')
print(f'  Recall      : {metrics.box.r.mean():.3f}')
print('='*45)

### 📉 Training Plots

In [ ]:
from IPython.display import Image, display
import os
results_dir = 'weapon_detection/yolov8m_cctv'
for fname, title in [('results.png','Training Curves'), ('confusion_matrix.png','Confusion Matrix'), ('PR_curve.png','PR Curve'), ('val_batch0_pred.jpg','Val Predictions')]:
    fpath = os.path.join(results_dir, fname)
    if os.path.exists(fpath):
        print(f'\n── {title} ──')
        display(Image(fpath, width=800))

### 🔍 Inference on Test Samples

In [ ]:
import glob, random, cv2, matplotlib.pyplot as plt, os
base = os.path.dirname(DATASET_YAML)
test_imgs = glob.glob(f'{base}/test/images/*.jpg') + glob.glob(f'{base}/valid/images/*.jpg')
samples = random.sample(test_imgs, min(4, len(test_imgs)))
fig, axes = plt.subplots(1, len(samples), figsize=(18, 5))
fig.suptitle('YOLOv8m Predictions', fontsize=14, fontweight='bold')
if len(samples) == 1: axes = [axes]
for ax, p in zip(axes, samples):
    r = best_model(p, conf=0.35, verbose=False)[0]
    img = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'{len(r.boxes)} det.')
plt.tight_layout(); plt.show()

## 💾 Step 7 — Save to Google Drive

In [ ]:
import shutil, os
weights_dir = 'weapon_detection/yolov8m_cctv/weights'
for wf in ['best.pt', 'last.pt']:
    src = os.path.join(weights_dir, wf)
    dst = os.path.join(SAVE_DIR, wf)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✅ {wf} → {dst}  ({os.path.getsize(dst)/1e6:.1f} MB)')

results_dst = os.path.join(SAVE_DIR, 'training_results')
shutil.copytree('weapon_detection/yolov8m_cctv', results_dst, dirs_exist_ok=True)
print(f'✅ Results → {results_dst}')
print('\n🎉 Done! Copy best.pt to Digital_Final/models/best.pt and restart the server.')

## 🧪 Bonus — Fine-tune on Your Own Camera Images
Label your CCTV footage for free at [app.roboflow.com](https://app.roboflow.com), export as YOLOv8, then:

In [ ]:
# Uncomment and fill in to fine-tune on your own data

# MY_DATA_YAML = "/path/to/your/data.yaml"
# ft = YOLO('weapon_detection/yolov8m_cctv/weights/best.pt')
# ft.train(
#     data=MY_DATA_YAML,
#     epochs=30,
#     imgsz=640,
#     batch=8,
#     lr0=0.0001,   # Lower LR for fine-tuning
#     freeze=10,    # Freeze backbone
#     project='weapon_detection',
#     name='yolov8m_finetune',
#     device=0,
# )
print('Fine-tune template ready. Uncomment to use.')

---
## 📦 Deploy to Aegis Threat Guard
```bash
cp ~/Downloads/best.pt Digital_Final/models/best.pt
python main.py
```
Then select `best.pt` in the **Settings page** of the dashboard.

---
*Notebook for Aegis Threat Guard — YOLOv8m CCTV Weapon Detection*